# CSCN8020 — Reinforcement Learning Programming
## Assignment 1

**Student:** Liggia Cruz 9085905  
**Course:** CSCN8020 — Reinforcement Learning Programming  

---

This notebook contains the complete solution for Assignment 1, covering four problems:

| Problem | Topic | Points |
|---------|-------|--------|
| 1 | Pick-and-Place Robot — MDP Design | 10 |
| 2 | 2×2 Gridworld — Value Iteration (manual) | 20 |
| 3 | 5×5 Gridworld — Value Iteration + In-Place Variation | 35 |
| 4 | Off-Policy Monte Carlo with Importance Sampling | 35 |

All algorithms are based on:  
> Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.

---

## Setup — Imports and Logger

The cell below imports all libraries used throughout the notebook and initialises the global logger.  
Run this cell first before executing any other cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import logging
import os
import time
import itertools
from pathlib import Path
from datetime import datetime

# ── Logger setup ──────────────────────────────────────────────────────────────
Path("logs").mkdir(exist_ok=True)
log_path = f"logs/execution_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(log_path),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("RL_Assignment1")
logger.info("=" * 60)
logger.info("CSCN8020 Assignment 1 — Execution started")
logger.info(f"Log file: {log_path}")
logger.info("=" * 60)

---
# Problem 1 — Pick-and-Place Robot MDP Design
**[10 points]**

---

## Background: What is a Markov Decision Process?

A **Markov Decision Process (MDP)** is the mathematical framework used to model sequential decision-making problems. An MDP is defined by the tuple:

$$\mathcal{M} = \langle \mathcal{S},\; \mathcal{A},\; P,\; R,\; \gamma \rangle$$

| Symbol | Meaning |
|--------|---------|
| $\mathcal{S}$ | Set of all possible **states** |
| $\mathcal{A}$ | Set of all possible **actions** |
| $P(s' \mid s, a)$ | **Transition probability** — likelihood of reaching $s'$ from $s$ after action $a$ |
| $R(s, a, s')$ | **Reward** received after transitioning from $s$ to $s'$ via $a$ |
| $\gamma \in [0,1)$ | **Discount factor** — how much future rewards are valued relative to immediate ones |

The **Markov property** requires that the future depends only on the current state:

$$P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \ldots) = P(s_{t+1} \mid s_t, a_t)$$

The agent's goal is to find a **policy** $\pi: \mathcal{S} \rightarrow \mathcal{A}$ that maximises the expected **return**:

$$G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

## Problem Statement

A robot arm must perform a repetitive **pick-and-place task**: grab an object from a fixed source and place it at a fixed target. The agent controls joint motors directly and receives feedback on positions and velocities.

Goal: learn movements that are **fast** (short duration) and **smooth** (low jerk).

---

## MDP Design

### State Space $\mathcal{S}$

For a robot arm with $n$ joints, the state at time $t$ is:

$$s_t = \bigl(\theta_1, \dot{\theta}_1,\; \theta_2, \dot{\theta}_2,\; \ldots,\; \theta_n, \dot{\theta}_n,\; g\bigr)$$

- $\theta_i \in [-\pi, \pi]$ — angular **position** of joint $i$ (radians)
- $\dot{\theta}_i$ — angular **velocity** of joint $i$ (rad/s)
- $g \in \{0, 1\}$ — **gripper**: 0 = open, 1 = closed

**Reasoning:** Both position and velocity are required — knowing only $\theta_i$ without $\dot{\theta}_i$ violates the Markov property because the next state depends on the current motion, not just the current angle.

---

### Action Space $\mathcal{A}$

$$a_t = \bigl(\Delta\dot{\theta}_1,\; \Delta\dot{\theta}_2,\; \ldots,\; \Delta\dot{\theta}_n,\; \delta_g\bigr)$$

- $\Delta\dot{\theta}_i \in \{-\Delta_{\max}, 0, +\Delta_{\max}\}$ — velocity increment per joint
- $\delta_g \in \{\text{open}, \text{close}\}$ — gripper command

---

### Transition Function $P(s' \mid s, a)$

$$\theta_i(t+1) = \theta_i(t) + \dot{\theta}_i(t)\,\Delta t \qquad \dot{\theta}_i(t+1) = \dot{\theta}_i(t) + \Delta\dot{\theta}_i$$

Transitions are near-deterministic; small Gaussian noise $\varepsilon \sim \mathcal{N}(0, \sigma^2)$ can model friction.

---

### Reward Function $R(s, a, s')$

$$R(s, a, s') = R_{\text{task}} + R_{\text{time}} + R_{\text{smooth}} + R_{\text{safety}}$$

| Component | Value | Purpose |
|-----------|-------|---------|
| $R_{\text{task}}$ | $+100$ on success, else $0$ | Reward placement |
| $R_{\text{time}}$ | $-1$ per timestep | Encourage speed |
| $R_{\text{smooth}}$ | $-\lambda\sum_i (\Delta\dot{\theta}_i)^2$ | Penalise jerk |
| $R_{\text{safety}}$ | $-50$ on collision/limit | Discourage unsafe moves |

### Discount Factor

We use $\gamma = 0.99$ so that the large terminal reward remains significant throughout long episodes:

$$G_t = R_{t+1} + 0.99\,R_{t+2} + 0.99^2\,R_{t+3} + \cdots$$

---

## Talking Points

### Talking Point A — Key RL Feature: Reward Shaping
The composite reward $R = R_{\text{task}} + R_{\text{time}} + R_{\text{smooth}} + R_{\text{safety}}$ is an instance of **reward shaping**: dense intermediate signals guide the agent without changing the optimal policy (Ng et al., 1999). No labelled data exists — the agent learns entirely from this scalar feedback.

### Talking Point B — Implementation Challenge: Continuous State Space
The joint positions and velocities form a continuous, high-dimensional state space. Standard tabular RL cannot be applied directly. Discretisation introduces a resolution trade-off: coarse grids lose precision, fine grids explode memory requirements (the *curse of dimensionality*).

### Talking Point C — Why This Is Reinforcement Learning
This is RL — not supervised or unsupervised learning — because (1) no correct action labels exist, (2) the learning signal is a scalar reward from the environment, and (3) the agent must solve the **temporal credit-assignment** problem: attributing the final $+100$ placement bonus to the sequence of motor commands that earned it. This maps directly to Sutton & Barto Figure 3.1 (agent–environment loop) and the Bellman optimality equations (Chapter 4).

## Code: MDP Representation

The class below encodes the MDP tuple $\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$ with a discretised state space.

In [ ]:
class PickAndPlaceMDP:
    """
    Discretised MDP for a 2-joint pick-and-place robot arm.

    State  : (theta1_idx, vel1_idx, theta2_idx, vel2_idx, gripper)
    Action : (delta_vel1, delta_vel2, gripper_cmd)
    """

    N_ANGLE = 5       # angle bins per joint  [-pi, -pi/2, 0, pi/2, pi]
    N_VEL   = 3       # velocity bins per joint  [-1, 0, +1] rad/s
    N_GRIP  = 2       # gripper states  {open=0, closed=1}

    DELTA_VEL_OPTIONS = [-1, 0, 1]
    GRIPPER_CMDS      = [0, 1]

    R_TASK   =  100.0
    R_TIME   =   -1.0
    R_SMOOTH =   -0.5   # lambda for smoothness penalty
    R_SAFETY =  -50.0
    GAMMA    =    0.99

    def __init__(self):
        self.states  = self._build_states()
        self.actions = self._build_actions()
        logger.info("[P1] PickAndPlaceMDP initialised")
        logger.info(f"     |S|={len(self.states)}  |A|={len(self.actions)}  gamma={self.GAMMA}")

    def _build_states(self):
        return list(itertools.product(
            range(self.N_ANGLE), range(self.N_VEL),
            range(self.N_ANGLE), range(self.N_VEL),
            range(self.N_GRIP)
        ))

    def _build_actions(self):
        return list(itertools.product(
            self.DELTA_VEL_OPTIONS,
            self.DELTA_VEL_OPTIONS,
            self.GRIPPER_CMDS
        ))

    def reward(self, action, placed=False, safety_violated=False):
        dv1, dv2, _ = action
        return (
            (self.R_TASK if placed else 0.0)
            + self.R_TIME
            + self.R_SMOOTH * (dv1**2 + dv2**2)
            + (self.R_SAFETY if safety_violated else 0.0)
        )

    def describe(self):
        print("=" * 50)
        print(" Pick-and-Place Robot MDP")
        print("=" * 50)
        print(f"  |S| = {len(self.states):>6} states")
        print(f"  |A| = {len(self.actions):>6} actions")
        print(f"  gamma          = {self.GAMMA}")
        print(f"  R_task         = +{self.R_TASK}  (on success)")
        print(f"  R_time         = {self.R_TIME}   (per step)")
        print(f"  R_smooth lam   = {self.R_SMOOTH}   (jerk penalty)")
        print(f"  R_safety       = {self.R_SAFETY}  (collision)")
        print("=" * 50)


mdp_p1 = PickAndPlaceMDP()
mdp_p1.describe()

# Sample: action = (accel j1, decel j2, close gripper)
action_sample = (1, -1, 1)
r_sample = mdp_p1.reward(action_sample, placed=False, safety_violated=False)
print(f"\nSample reward for action {action_sample}: {r_sample}")
logger.info(f"[P1] Sample reward: {r_sample}")

---
# Problem 2 — 2×2 Gridworld: Value Iteration
**[20 points]**

---

## Environment

```
┌─────────────┬─────────────┐
│  s1  R = 5  │  s2  R = 10 │
├─────────────┼─────────────┤
│  s3  R = 1  │  s4  R = 2  │
└─────────────┴─────────────┘
```

- **States** $\mathcal{S} = \{s_1, s_2, s_3, s_4\}$
- **Actions** $\mathcal{A} = \{\text{up}, \text{down}, \text{left}, \text{right}\}$
- **Initial policy** $\pi_0$: $\pi(\text{up} \mid s) = 1$ for all $s$
- **Transitions**: deterministic when valid; otherwise $s' = s$ (wall bounce)
- **Rewards**: $R(s_1)=5,\; R(s_2)=10,\; R(s_3)=1,\; R(s_4)=2$ (state-dependent, action-independent)
- **Discount factor** $\gamma = 0.9$ *(assumed, not specified in the original problem)*

### Full Transition Table

| State | up | down | left | right |
|-------|-----|------|------|-------|
| $s_1$ (top-left)     | $s_1$ (wall) | $s_3$ | $s_1$ (wall) | $s_2$ |
| $s_2$ (top-right)    | $s_2$ (wall) | $s_4$ | $s_1$        | $s_2$ (wall) |
| $s_3$ (bottom-left)  | $s_1$        | $s_3$ (wall) | $s_3$ (wall) | $s_4$ |
| $s_4$ (bottom-right) | $s_2$        | $s_4$ (wall) | $s_3$        | $s_4$ (wall) |

## Theory: Value Iteration

**Value Iteration** (Sutton & Barto, 2018, §4.4) finds the optimal value function $V^*$ by repeatedly applying the **Bellman optimality operator**:

$$V_{k+1}(s) = \max_{a \in \mathcal{A}} \left[ R(s) + \gamma \sum_{s'} P(s' \mid s, a)\, V_k(s') \right]$$

Because transitions here are **deterministic**, $P(s' \mid s, a) = 1$ for exactly one $s'$:

$$\boxed{V_{k+1}(s) = R(s) + \gamma\, \max_{a \in \mathcal{A}}\, V_k\!\left(T(s,a)\right)}$$

where $T(s, a)$ is the next state reached from $s$ by action $a$.

Since $R(s)$ does not depend on $a$, it factors out of the $\max$.

The **greedy policy** extracted after each sweep is:

$$\pi_{k+1}(s) = \arg\max_{a \in \mathcal{A}} V_k\!\left(T(s,a)\right)$$

### Pseudocode (Sutton & Barto, p. 83)

```
Input: MDP(S, A, P, R, γ), threshold θ
Initialise V(s) = 0 for all s ∈ S
Loop:
    Δ ← 0
    For each s ∈ S:
        v ← V(s)
        V(s) ← max_a [ R(s) + γ Σ_{s'} P(s'|s,a) V(s') ]
        Δ ← max(Δ, |v − V(s)|)
Until Δ < θ
Output: π*(s) = argmax_a [ R(s) + γ Σ_{s'} P(s'|s,a) V(s') ]
```

## Manual Step-by-Step: Iteration 1

### Initial Value Function

$$V_0(s_1) = 0 \qquad V_0(s_2) = 0 \qquad V_0(s_3) = 0 \qquad V_0(s_4) = 0$$

### Value Function Updates

We apply $V_1(s) = R(s) + \gamma\,\max_a V_0(T(s,a))$. Because $V_0 = 0$ everywhere, the $\max$ term is always $0$.

$$V_1(s_1) = R(s_1) + 0.9 \cdot \max\bigl(V_0(s_1),\, V_0(s_3),\, V_0(s_1),\, V_0(s_2)\bigr)$$
$$= 5 + 0.9 \cdot \max(0, 0, 0, 0) = 5 + 0 = \mathbf{5}$$

$$V_1(s_2) = R(s_2) + 0.9 \cdot \max\bigl(V_0(s_2),\, V_0(s_4),\, V_0(s_1),\, V_0(s_2)\bigr)$$
$$= 10 + 0.9 \cdot \max(0, 0, 0, 0) = 10 + 0 = \mathbf{10}$$

$$V_1(s_3) = R(s_3) + 0.9 \cdot \max\bigl(V_0(s_1),\, V_0(s_3),\, V_0(s_3),\, V_0(s_4)\bigr)$$
$$= 1 + 0.9 \cdot \max(0, 0, 0, 0) = 1 + 0 = \mathbf{1}$$

$$V_1(s_4) = R(s_4) + 0.9 \cdot \max\bigl(V_0(s_2),\, V_0(s_4),\, V_0(s_3),\, V_0(s_4)\bigr)$$
$$= 2 + 0.9 \cdot \max(0, 0, 0, 0) = 2 + 0 = \mathbf{2}$$

### Updated Value Function after Iteration 1

$$\boxed{V_1(s_1) = 5 \qquad V_1(s_2) = 10 \qquad V_1(s_3) = 1 \qquad V_1(s_4) = 2}$$

```
┌──────┬──────┐
│  5.0 │ 10.0 │
├──────┼──────┤
│  1.0 │  2.0 │
└──────┴──────┘
```

### Greedy Policy after Iteration 1

$$\pi_1(s) = \arg\max_a V_1(T(s,a))$$

| State | up | down | left | right | **Best action** |
|-------|----|------|------|-------|----------------|
| $s_1$ | $V_1(s_1)=5$ | $V_1(s_3)=1$ | $V_1(s_1)=5$ | $V_1(s_2)=\mathbf{10}$ | **right** |
| $s_2$ | $V_1(s_2)=\mathbf{10}$ | $V_1(s_4)=2$ | $V_1(s_1)=5$ | $V_1(s_2)=\mathbf{10}$ | **up** (tie with right) |
| $s_3$ | $V_1(s_1)=\mathbf{5}$ | $V_1(s_3)=1$ | $V_1(s_3)=1$ | $V_1(s_4)=2$ | **up** |
| $s_4$ | $V_1(s_2)=\mathbf{10}$ | $V_1(s_4)=2$ | $V_1(s_3)=1$ | $V_1(s_4)=2$ | **up** |

## Manual Step-by-Step: Iteration 2

We apply $V_2(s) = R(s) + 0.9\,\max_a V_1(T(s,a))$, now using $V_1$ from the previous step.

$$V_2(s_1) = R(s_1) + 0.9 \cdot \max\bigl(V_1(s_1),\, V_1(s_3),\, V_1(s_1),\, V_1(s_2)\bigr)$$
$$= 5 + 0.9 \cdot \max(5,\; 1,\; 5,\; \mathbf{10}) = 5 + 0.9 \times 10 = 5 + 9 = \mathbf{14}$$

$$V_2(s_2) = R(s_2) + 0.9 \cdot \max\bigl(V_1(s_2),\, V_1(s_4),\, V_1(s_1),\, V_1(s_2)\bigr)$$
$$= 10 + 0.9 \cdot \max(\mathbf{10},\; 2,\; 5,\; \mathbf{10}) = 10 + 0.9 \times 10 = 10 + 9 = \mathbf{19}$$

$$V_2(s_3) = R(s_3) + 0.9 \cdot \max\bigl(V_1(s_1),\, V_1(s_3),\, V_1(s_3),\, V_1(s_4)\bigr)$$
$$= 1 + 0.9 \cdot \max(\mathbf{5},\; 1,\; 1,\; 2) = 1 + 0.9 \times 5 = 1 + 4.5 = \mathbf{5.5}$$

$$V_2(s_4) = R(s_4) + 0.9 \cdot \max\bigl(V_1(s_2),\, V_1(s_4),\, V_1(s_3),\, V_1(s_4)\bigr)$$
$$= 2 + 0.9 \cdot \max(\mathbf{10},\; 2,\; 1,\; 2) = 2 + 0.9 \times 10 = 2 + 9 = \mathbf{11}$$

### Updated Value Function after Iteration 2

$$\boxed{V_2(s_1) = 14 \qquad V_2(s_2) = 19 \qquad V_2(s_3) = 5.5 \qquad V_2(s_4) = 11}$$

```
┌──────┬──────┐
│ 14.0 │ 19.0 │
├──────┼──────┤
│  5.5 │ 11.0 │
└──────┴──────┘
```

### Greedy Policy after Iteration 2

| State | up | down | left | right | **Best action** |
|-------|----|------|------|-------|----------------|
| $s_1$ | $V_2(s_1)=14$ | $V_2(s_3)=5.5$ | $V_2(s_1)=14$ | $V_2(s_2)=\mathbf{19}$ | **right** |
| $s_2$ | $V_2(s_2)=\mathbf{19}$ | $V_2(s_4)=11$ | $V_2(s_1)=14$ | $V_2(s_2)=\mathbf{19}$ | **up** (tie with right) |
| $s_3$ | $V_2(s_1)=\mathbf{14}$ | $V_2(s_3)=5.5$ | $V_2(s_3)=5.5$ | $V_2(s_4)=11$ | **up** |
| $s_4$ | $V_2(s_2)=\mathbf{19}$ | $V_2(s_4)=11$ | $V_2(s_3)=5.5$ | $V_2(s_4)=11$ | **up** |

**Intuition:** $s_2$ has the highest reward ($R=10$) and the policy is already converging toward reaching $s_2$ from every state and staying there.

## Code Verification

The classes below implement the 2×2 gridworld environment and a Value Iteration agent.  
Running two iterations must produce the same values computed manually above.

In [ ]:
class GridWorld2x2:
    """
    2x2 gridworld MDP.

    States (linear index):
        0 = s1 (top-left,  R=5 )
        1 = s2 (top-right, R=10)
        2 = s3 (bot-left,  R=1 )
        3 = s4 (bot-right, R=2 )

    Actions: 0=up, 1=down, 2=left, 3=right
    """

    N_STATES  = 4
    N_ACTIONS = 4
    ACTION_NAMES = ["up", "down", "left", "right"]

    # R(s) for each state index
    REWARDS = np.array([5.0, 10.0, 1.0, 2.0])

    def __init__(self, gamma: float = 0.9):
        self.gamma = gamma
        # transitions[s][a] = s'
        self.transitions = self._build_transitions()
        logger.info("[P2] GridWorld2x2 initialised  gamma=%.2f", gamma)

    def _build_transitions(self) -> np.ndarray:
        """
        Build the deterministic transition table.
        transitions[s, a] = next_state
        """
        # Grid positions: state_idx -> (row, col)
        pos = {0: (0, 0), 1: (0, 1), 2: (1, 0), 3: (1, 1)}
        # Reverse map: (row, col) -> state_idx
        idx = {v: k for k, v in pos.items()}

        # Deltas for [up, down, left, right]
        deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        T = np.zeros((self.N_STATES, self.N_ACTIONS), dtype=int)
        for s in range(self.N_STATES):
            r, c = pos[s]
            for a, (dr, dc) in enumerate(deltas):
                nr, nc = r + dr, c + dc
                # If new position is within the 2x2 grid, move; else stay
                T[s, a] = idx.get((nr, nc), s)
        return T

    def step(self, state: int, action: int):
        """Return (next_state, reward)."""
        next_state = self.transitions[state, action]
        reward = self.REWARDS[state]   # reward depends on current state
        return next_state, reward


class ValueIterationAgent2x2:
    """
    Runs exactly n_iterations sweeps of the Bellman optimality update.

    Maps to Sutton & Barto (2018) Algorithm: Value Iteration (p. 83).
    """

    def __init__(self, env: GridWorld2x2):
        self.env = env
        self.V   = np.zeros(env.N_STATES)   # V_0 = 0 for all states
        self.policy = np.zeros(env.N_STATES, dtype=int)

    def _bellman_update(self) -> float:
        """
        One synchronous sweep: V_{k+1}(s) = R(s) + gamma * max_a V_k(T(s,a)).
        Returns the maximum change (delta) across all states.
        """
        V_new = np.zeros(self.env.N_STATES)
        for s in range(self.env.N_STATES):
            # Q-values for all actions from state s
            q_values = [
                self.env.REWARDS[s] + self.env.gamma * self.V[self.env.transitions[s, a]]
                for a in range(self.env.N_ACTIONS)
            ]
            V_new[s] = max(q_values)
        delta = np.max(np.abs(V_new - self.V))
        self.V = V_new
        return delta

    def _extract_policy(self):
        """Greedy policy: pi(s) = argmax_a V(T(s,a))."""
        for s in range(self.env.N_STATES):
            q_values = [
                self.env.REWARDS[s] + self.env.gamma * self.V[self.env.transitions[s, a]]
                for a in range(self.env.N_ACTIONS)
            ]
            self.policy[s] = int(np.argmax(q_values))

    def run(self, n_iterations: int):
        """Run exactly n_iterations Bellman sweeps and print results."""
        logger.info("[P2] ValueIterationAgent2x2 — starting %d iterations", n_iterations)
        for k in range(1, n_iterations + 1):
            delta = self._bellman_update()
            self._extract_policy()
            logger.info("  Iteration %d | delta=%.4f | V=%s", k, delta, np.round(self.V, 4))
            print(f"\n--- Iteration {k} ---")
            self._print_results(k)

    def _print_results(self, k: int):
        names = ["s1", "s2", "s3", "s4"]
        action_names = self.env.ACTION_NAMES
        print("  Value function V_{}:".format(k))
        for i, name in enumerate(names):
            print(f"    V({name}) = {self.V[i]:.1f}")
        print("  Gridworld layout:")
        print(f"    | {self.V[0]:5.1f} | {self.V[1]:5.1f} |")
        print(f"    | {self.V[2]:5.1f} | {self.V[3]:5.1f} |")
        print("  Greedy policy pi_{}:".format(k))
        for i, name in enumerate(names):
            print(f"    pi({name}) = {action_names[self.policy[i]]}")


# ── Run 2 iterations and verify against manual calculations ───────────────────
env_2x2   = GridWorld2x2(gamma=0.9)
agent_2x2 = ValueIterationAgent2x2(env_2x2)
agent_2x2.run(n_iterations=2)

# Verify against expected manual values
expected_V2 = np.array([14.0, 19.0, 5.5, 11.0])
assert np.allclose(agent_2x2.V, expected_V2), "Mismatch with manual calculation!"
print("\n✓ Code output matches manual calculation exactly.")
logger.info("[P2] Verification passed — code matches manual values.")

## Talking Points

### Talking Point A — Key RL Feature: Bellman Optimality and Policy Improvement

The central mechanism in Value Iteration is the **Bellman optimality operator** $\mathcal{T}^*$:

$$V_{k+1}(s) = \mathcal{T}^* V_k(s) = R(s) + \gamma\,\max_{a}\, V_k(T(s,a))$$

Each sweep simultaneously performs a **truncated policy evaluation** (computing $V$ for the current greedy policy) and **policy improvement** (updating the greedy policy). This is why Value Iteration converges to $V^*$ even though no full policy evaluation is done at each step — the Bellman operator is a contraction mapping (Banach fixed-point theorem), guaranteeing $\|V_{k+1} - V^*\|_\infty \leq \gamma \|V_k - V^*\|_\infty$.

### Talking Point B — Implementation Challenge: Matching Code Output with Manual Results

A subtle challenge was ensuring the reward convention is consistent: in this problem $R(s)$ is earned when **leaving** state $s$ (not when entering the next state $s'$). The Bellman update $V(s) = R(s) + \gamma\max_a V(T(s,a))$ reflects this — the reward is tied to the current state, not the transition target. Using $R(s')$ instead would produce different (incorrect) values. The `assert` statement in the code cell above automatically catches any such mismatch.

### Talking Point C — Why This Is Reinforcement Learning

This solution is RL because it learns the **value function** through interaction with an environment model (the MDP), without being given the optimal policy directly. The **policy improvement theorem** (Sutton & Barto, §4.2) guarantees that the greedy policy $\pi_{k+1}$ is at least as good as $\pi_k$, and repeated application converges to $\pi^*$. The key RL concepts are all present: **state** ($s \in \mathcal{S}$), **action** ($a \in \mathcal{A}$), **reward** ($R(s)$), **value function** ($V_k$), and **Bellman update** (the sweep). This contrasts with supervised learning, which would require pre-labelled optimal actions for each state.

---
# Problem 3 — 5×5 Gridworld: Value Iteration Variations
**[35 points]**

---

## Environment

```
     c0    c1    c2    c3    c4
r0 [  -1][  -1][  -1][  -1][-5*]
r1 [  -1][  -1][  -1][  -1][  -1]
r2 [  -1][  -1][-5*][  -1][  -1]
r3 [-5*][  -1][  -1][  -1][  -1]
r4 [  -1][  -1][  -1][  -1][GOAL]
```

*Grey states (marked with `*`): $S_{\text{grey}} = \{s_{2,2},\; s_{3,0},\; s_{0,4}\}$*

| State type | Condition | Reward on entry |
|------------|-----------|----------------|
| Terminal / Goal | $s = s_{4,4}$ | $+10$ |
| Grey | $s \in S_{\text{grey}}$ | $-5$ |
| Regular | otherwise | $-1$ |

- **Actions**: $a_1 = \text{right},\; a_2 = \text{down},\; a_3 = \text{left},\; a_4 = \text{up}$
- **Transitions**: deterministic; wall actions keep agent in place: $s' = s$
- **Discount factor**: $\gamma = 0.9$
- **Terminal condition**: episode ends when the agent enters $s_{4,4}$; $V^*(s_{4,4}) = 0$

### Reward Convention
Reward is received upon **entering** a state $s'$:

$$R(s, a) = R_{\text{entry}}\bigl(T(s,a)\bigr)$$

So the Bellman optimality update becomes:

$$\boxed{V_{k+1}(s) = \max_{a \in \mathcal{A}} \Bigl[ R_{\text{entry}}\!\bigl(T(s,a)\bigr) + \gamma\, V_k\!\bigl(T(s,a)\bigr) \Bigr]}$$

and the terminal state is fixed: $V(s_{4,4}) = 0$ (not updated during sweeps).


## Theory: Standard VI vs In-Place VI

### Task 1 — Standard (Synchronous) Value Iteration

Each sweep computes a **frozen copy** $V_{\text{new}}$ from $V_{\text{old}}$:

$$V_{k+1}(s) \leftarrow \max_a \bigl[R_{\text{entry}}(T(s,a)) + \gamma\,V_k(T(s,a))\bigr] \quad \forall s$$

All states in one sweep use the same (old) values — the update is *synchronous*.

### Task 2 — In-Place Value Iteration

A **single array** $V$ is updated in-place. When state $s_j$ is updated after $s_i$ in the same sweep, the already-updated $V(s_i)$ may be used:

$$V(s) \leftarrow \max_a \bigl[R_{\text{entry}}(T(s,a)) + \gamma\,V(T(s,a))\bigr]$$

where $V(T(s,a))$ may already reflect updates from the current sweep.

### Pseudocode Comparison (Sutton & Barto, 2018, p. 83)

```
Standard VI (synchronous):          In-Place VI:
────────────────────────────────    ───────────────────────────────
V_old ← copy(V)                    (no copy needed)
for each s ∈ S:                     for each s ∈ S:
    V_new[s] ← max_a Q(s,a,V_old)      V[s] ← max_a Q(s,a,V)   ← uses live V
V ← V_new                          (update in place — no swap)
```

### Complexity

| Variant | Memory | Convergence |
|---------|--------|-------------|
| Standard VI | $O(2|S|)$ — two arrays | More iterations (uses stale values) |
| In-Place VI | $O(|S|)$ — one array | Fewer iterations (uses fresh values) |

Both variants have the same per-iteration complexity $O(|S| \cdot |A|)$ and are guaranteed to converge to the same $V^*$ and $\pi^*$.


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())   # allow `from src.xxx import ...`

from src.environments import GridWorld5x5
from src.agents import ValueIterationAgent, InPlaceValueIterationAgent
from src.utils import plot_gridworld, print_value_table, print_policy_table
from src.policies import Policy

# ── Create environment ────────────────────────────────────────────────────────
GAMMA = 0.9
env_5x5 = GridWorld5x5(gamma=GAMMA)

logger.info("[P3] GridWorld5x5 created: %d states, %d actions, gamma=%.1f",
            env_5x5.n_states, env_5x5.n_actions, GAMMA)

print("Environment summary:")
print(f"  Grid size   : {env_5x5.ROWS} x {env_5x5.COLS}")
print(f"  Goal state  : {env_5x5.GOAL}  (index {env_5x5.goal_idx})")
print(f"  Grey states : {sorted(env_5x5.GREY_STATES)}")
print(f"  R_goal={env_5x5.R_GOAL}  R_grey={env_5x5.R_GREY}  R_regular={env_5x5.R_REGULAR}")
print(f"  gamma = {env_5x5.gamma}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Task 1: Standard (Synchronous) Value Iteration
# ─────────────────────────────────────────────────────────────────────────────
logger.info("[P3] ===== Standard Value Iteration =====")

vi_agent = ValueIterationAgent(env_5x5, theta=1e-6)
vi_iters, vi_time = vi_agent.run()

print(f"\nStandard VI converged in {vi_iters} iterations  ({vi_time*1000:.2f} ms)")
print_value_table(env_5x5, vi_agent.V, label="V* — Standard VI")
print_policy_table(env_5x5, vi_agent.policy)
plot_gridworld(env_5x5, vi_agent.V, vi_agent.policy, title="Standard_VI")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Task 2: In-Place Value Iteration
# ─────────────────────────────────────────────────────────────────────────────
logger.info("[P3] ===== In-Place Value Iteration =====")

ip_agent = InPlaceValueIterationAgent(env_5x5, theta=1e-6)
ip_iters, ip_time = ip_agent.run()

print(f"In-Place VI converged in {ip_iters} iterations  ({ip_time*1000:.2f} ms)")
print_value_table(env_5x5, ip_agent.V, label="V* — In-Place VI")
print_policy_table(env_5x5, ip_agent.policy)
plot_gridworld(env_5x5, ip_agent.V, ip_agent.policy, title="In_Place_VI")

# ── Verification: both must reach the same V* and pi* ────────────────────────
assert np.allclose(vi_agent.V, ip_agent.V, atol=1e-4), \
    f"V* mismatch!\nVI:   {vi_agent.V}\nIP:   {ip_agent.V}"
assert np.array_equal(vi_agent.policy, ip_agent.policy), \
    "Policy mismatch between Standard VI and In-Place VI!"
print("\n✓  Both algorithms converged to the same V* and pi*\n")
logger.info("[P3] Verification passed — VI and IP-VI match.")

# ── Performance comparison table ──────────────────────────────────────────────
print("=" * 55)
print("  Performance Comparison: Standard VI vs In-Place VI")
print("=" * 55)
print(f"  {'Algorithm':<22} {'Iterations':>12} {'Time (ms)':>12}")
print(f"  {'-'*46}")
print(f"  {'Standard VI':<22} {vi_iters:>12} {vi_time*1000:>12.2f}")
print(f"  {'In-Place VI':<22} {ip_iters:>12} {ip_time*1000:>12.2f}")
print("=" * 55)
ratio = vi_iters / max(ip_iters, 1)
print(f"\n  In-Place VI required {ratio:.1f}x fewer iterations.")
print(f"  Memory: Standard VI uses 2 arrays; In-Place VI uses 1.")
logger.info("[P3] Standard VI: %d iter  In-Place VI: %d iter  ratio=%.2f",
            vi_iters, ip_iters, ratio)


## Talking Points

### Talking Point A — Key RL Feature: Value Iteration as Bellman Optimality Operator

The update rule in both variants is an application of the **Bellman optimality operator** $\mathcal{T}^*$:

$$V_{k+1} = \mathcal{T}^* V_k, \quad \text{where } (\mathcal{T}^* V)(s) = \max_{a} \bigl[R(T(s,a)) + \gamma\,V(T(s,a))\bigr]$$

$\mathcal{T}^*$ is a **$\gamma$-contraction** in the $\ell^\infty$ norm (Banach fixed-point theorem):

$$\|V_{k+1} - V^*\|_\infty \;\leq\; \gamma\;\|V_k - V^*\|_\infty$$

This guarantees geometric convergence to the unique fixed point $V^*$ regardless of the initial values.

### Talking Point B — Implementation Challenge: Handling the Terminal State

A key challenge was correctly implementing the terminal state $s_{4,4}$. In an episodic task, the episode ends upon entering the goal — no future rewards are collected from it. If the terminal state is treated as a regular state in the Bellman update, it creates a self-referential loop:

$$V(s_{4,4}) = R(s_{4,4}) + \gamma V(s_{4,4}) \implies V(s_{4,4}) = \frac{R}{1-\gamma} = 100$$

which inflates values throughout the grid incorrectly. The fix — skipping the terminal state during sweeps and fixing $V(s_{4,4}) = 0$ — is essential for episodic task correctness and was verified by inspecting the neighbouring state $s_{4,3}$: its value converges to $V^*(s_{4,3}) \approx 9$ (one step away from the $+10$ goal reward with $\gamma=0.9$).

### Talking Point C — Why This Is Reinforcement Learning

Value Iteration is RL because it estimates the **optimal value function** $V^*$ by direct experience with the MDP model, without any labelled training data. It embodies core RL concepts from Sutton & Barto (2018):

- **State** $s \in \mathcal{S}$: the grid cell $(r, c)$
- **Action** $a \in \mathcal{A}$: move up/down/left/right
- **Reward** $R$: received on state entry (signal from the environment)
- **Value function** $V^*(s)$: expected discounted return from $s$ under the optimal policy
- **Bellman update**: the repeated application of $\mathcal{T}^*$ (Chapter 4)
- **Policy improvement**: the greedy policy $\pi^* = \arg\max_a Q^*(s,a)$ is extracted once $V^*$ is found

The In-Place variant additionally relates to **asynchronous DP** (Sutton & Barto §4.5), where states are updated with the latest available information rather than waiting for a full synchronous sweep.


---
# Problem 4 — Off-Policy Monte Carlo with Importance Sampling
**[35 points]**

---

## Problem Statement

Using the **same 5×5 gridworld** from Problem 3, we implement **Off-Policy Monte Carlo (MC) Control**
with **Weighted Importance Sampling** to estimate $Q^*$, $V^*$, and $\pi^*$ without accessing the
environment model.

| | Value Iteration (P3) | Off-Policy MC (P4) |
|--|--|--|
| Requires MDP model? | **Yes** ($P$, $R$ tables) | **No** (model-free) |
| Learning signal | Bellman equation | Sampled returns $G_t$ |
| Convergence | Exact $V^*$ | Approximation (improves with more episodes) |

---

## Setup

- **Behavior policy** $b(a \mid s) = \dfrac{1}{|\mathcal{A}|} = 0.25$ — uniform random (guarantees *coverage*)
- **Target policy** $\pi(s) = \arg\max_a Q(s,a)$ — greedy, updated each episode
- **Discount factor** $\gamma = 0.9$,  $N = 50{,}000$ episodes,  `max_steps` $= 500$

---

## Theory: Weighted Importance Sampling

### The Return

For an episode $S_0,A_0,R_1,\ldots,S_T$, the return at step $t$ is:

$$G_t = \sum_{k=0}^{T-t-1} \gamma^k\, R_{t+k+1}$$

### Importance Sampling Ratio

Corrections for the mismatch between $b$ (behavior) and $\pi$ (target):

$$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k \mid S_k)}{b(A_k \mid S_k)}$$

Since $\pi$ is deterministic and $b(a|s) = 1/4$:

$$\text{If } A_k = \pi(S_k): \quad \frac{\pi}{b} = \frac{1}{1/4} = 4$$
$$\text{If } A_k \neq \pi(S_k): \quad \frac{\pi}{b} = \frac{0}{1/4} = 0 \implies W = 0 \implies \textbf{break}$$

### Weighted IS Update (Sutton & Barto, §5.7)

Weighted IS has **lower variance** than ordinary IS at the cost of slight bias (bias $\to 0$ as $N \to \infty$):

$$C(s,a) \leftarrow C(s,a) + W$$
$$Q(s,a) \leftarrow Q(s,a) + \frac{W}{C(s,a)}\bigl[G - Q(s,a)\bigr]$$

### Pseudocode (Sutton & Barto, 2018, p. 110)

```
Initialise Q(s,a)=0, C(s,a)=0, pi(s)=argmax_a Q(s,a) for all s,a
Loop for each episode:
    Generate episode using b:  S0,A0,R1, ..., S_{T-1},A_{T-1},R_T
    G <- 0;  W <- 1
    For t = T-1, T-2, ..., 0:
        G <- gamma*G + R_{t+1}
        C(St,At) <- C(St,At) + W
        Q(St,At) <- Q(St,At) + (W/C(St,At)) * (G - Q(St,At))
        pi(St)   <- argmax_a Q(St,a)
        If At != pi(St): break          (IS weight tail = 0)
        W <- W * (1 / b(At|St))        (= W * |A| = W * 4)
Output: pi*, V*(s) = max_a Q*(s,a)
```


In [ ]:
from src.agents import MonteCarloOffPolicyAgent

logger.info('[P4] ===== Off-Policy Monte Carlo with IS =====')

In [ ]:
# ── Run off-policy MC ────────────────────────────────────────────────────
N_EPISODES = 50_000

mc_agent = MonteCarloOffPolicyAgent(
    env_5x5,
    gamma      = GAMMA,
    n_episodes = N_EPISODES,
    max_steps  = 500,
    seed       = 42
)
mc_episodes, mc_time = mc_agent.run()

print(f'MC finished: {mc_episodes:,} episodes  ({mc_time:.3f} s)')
print_value_table(env_5x5, mc_agent.V, label='V* -- Off-Policy MC')
print_policy_table(env_5x5, mc_agent.policy)
plot_gridworld(env_5x5, mc_agent.V, mc_agent.policy, title='Off_Policy_MC')

In [ ]:
# ── Compare MC vs VI ─────────────────────────────────────────────────────
v_diff   = np.abs(mc_agent.V - vi_agent.V)
mae      = float(np.mean(v_diff))
max_err  = float(np.max(v_diff))

non_term    = [s for s in range(env_5x5.n_states) if not env_5x5.is_goal(s)]
pol_agree   = np.mean(mc_agent.policy[non_term] == vi_agent.policy[non_term])

print('=' * 66)
print('  Off-Policy MC  vs  Standard Value Iteration')
print('=' * 66)
print(f"  {'Metric':<30} {'MC':>15} {'VI':>15}")
print(f"  {'-'*60}")
print(f"  {'Episodes / Iterations':<30} {mc_episodes:>15,} {vi_iters:>15,}")
print(f"  {'Wall-clock time (s)':<30} {mc_time:>15.3f} {vi_time:>15.4f}")
print(f"  {'Needs env model?':<30} {'No':>15} {'Yes':>15}")
print(f"  {'MAE vs VI ground truth':<30} {mae:>15.4f} {'0.0000':>15}")
print(f"  {'Max |V_mc - V_vi|':<30} {max_err:>15.4f} {'0.0000':>15}")
print(f"  {'Policy agreement with VI':<30} {pol_agree:>14.1%} {'100.0%':>15}")
print('=' * 66)

# Per-state error heatmap
fig, ax = plt.subplots(figsize=(5, 4.5))
diff_grid = v_diff.reshape(env_5x5.ROWS, env_5x5.COLS)
im = ax.imshow(diff_grid, cmap='YlOrRd', vmin=0)
plt.colorbar(im, ax=ax, label='|V_mc - V_vi|')
for s in range(env_5x5.n_states):
    r, c = env_5x5.to_pos(s)
    ax.text(c, r, f'{v_diff[s]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('Per-State Error: |V_MC - V_VI|')
ax.set_xticks(range(env_5x5.COLS)); ax.set_yticks(range(env_5x5.ROWS))
plt.tight_layout()
plt.savefig('images/MC_vs_VI_error.png', dpi=130, bbox_inches='tight')
plt.show()

logger.info('[P4] Comparison: MAE=%.4f  max_err=%.4f  policy_agree=%.1f%%',
            mae, max_err, pol_agree * 100)

## Talking Points

### Talking Point A — Key RL Feature: Behavior Policy vs Target Policy (Off-Policy Learning)

The defining feature of off-policy MC is the **separation of two roles**:

- **Behavior policy** $b(a \mid s) = 1/4$ — explores freely, generates episodes. Must satisfy *coverage*: $\pi(a \mid s) > 0 \Rightarrow b(a \mid s) > 0$.
- **Target policy** $\pi(s) = \arg\max_a Q(s,a)$ — the policy being improved toward $\pi^*$.

The **importance sampling ratio** $\rho = \prod_k \pi(A_k|S_k)/b(A_k|S_k)$ reweights the returns generated by $b$ so they are consistent with $\pi$. **Weighted IS** reduces variance by normalizing:

$$Q(s,a) \leftarrow Q(s,a) + \frac{W}{C(s,a)}\bigl[G - Q(s,a)\bigr], \qquad C(s,a) \leftarrow C(s,a) + W$$

This is an incremental implementation of the weighted average, which is unbiased asymptotically and has much lower variance than ordinary IS (Sutton & Barto, §5.6–5.7).

### Talking Point B — Implementation Challenge: IS Weight Explosion and Episode Coverage

Two challenges required careful design:

1. **IS weight growth**: When all actions in the backward pass match $\pi$, $W$ multiplies by $1/b = 4$ at each step. For a trajectory of length $L$, $W$ can reach $4^L$, causing numerical overflow. Weighted IS mitigates this because $W/C(s,a)$ is bounded — but very long episodes still risk precision issues. Setting `max_steps=500` was necessary to keep training stable.

2. **Sparse goal-reaching**: The uniform random policy rarely reaches $s_{4,4}$ on long paths. States far from the goal receive fewer useful IS-corrected updates, causing higher MC estimation error there (visible in the per-state heatmap). This is a known limitation of episodic MC methods — model-based methods (VI) do not suffer from it because they update all states every sweep regardless of episode structure.

### Talking Point C — Why This Is Reinforcement Learning and How It Differs from VI

MC is RL because (Sutton & Barto, Chapters 3, 5):

| Concept | MC (P4) | VI (P3) |
|---------|---------|---------|
| **Agent-environment loop** | Yes — samples episodes | No — queries the model |
| **Reward signal** | $R_{t+1}$ from environment | $R(s)$ from lookup table |
| **Learning from experience** | Yes — returns $G_t$ | No — computes $V$ analytically |
| **Model-free** | Yes | No (needs $P$, $R$) |
| **Bellman equation** | Approximate (sampled) | Exact (exhaustive) |

MC connects to Sutton & Barto's **Monte Carlo prediction** (§5.1) and **off-policy control** (§5.7). The key RL concept it illustrates is **learning from complete episodes** — the agent must finish an episode before updating $Q$, unlike TD methods (Chapter 6) which update at every step. This makes MC exact per sample (no bootstrapping) but slow to converge for long episodes.
